In [105]:
import pandas as pd
import numpy as np

In [106]:
emails = pd.read_csv('../../data/raw/emails.csv', low_memory=False)

emails.shape

(123389, 27)

Standardizing Column Names

In [107]:
emails.columns = (emails.columns.str.lower())
emails.columns

Index(['co_ref', 'time_to_renewal', 'crm_accreditation_completed',
       'crm_timely_completion', 'crm_progress_towards_accreditation',
       'crm_delays_in_accreditation', 'crm_contractor_suggested_leave',
       'crm_contractor_engagement', 'crm_contractor_sentiment',
       'crm_contractor_sentiment_score', 'crm_dts_or_ssip_mentioned',
       'crm_customer_payment_intention', 'crm_competitors_mentioned',
       'crm_membership_level', 'crm_platform_issues_raised',
       'crm_agent_chased_contractor', 'crm_agent_chase_count',
       'crm_accreditation_issues', 'crm_membership_overdue',
       'crm_auto_renewal_status', 'crm_dissatisified_with_renewal_price',
       'crm_customer_complained', 'crm_refund_mentioned',
       'crm_negative_customer_experience', 'crm_dissatisfaction_with_support',
       'crm_financial_hardship_mentioned', 'year'],
      dtype='str')

Filtering Churn Cases (14_out)

In [108]:
emails['time_to_renewal'].value_counts()

time_to_renewal
prior_year     40022
14_out         32493
45_out         28008
pre_renewal    22866
Name: count, dtype: int64

In [109]:
emails= emails[emails["time_to_renewal"] == "14_out"]

Checking duplicates

In [110]:
emails.duplicated().sum()

np.int64(0)

In [111]:
emails['co_ref'].isnull().sum()

np.int64(0)

In [112]:
emails['time_to_renewal'].value_counts()

time_to_renewal
14_out    32493
Name: count, dtype: int64

Dropping irrelevant columns<br>
year -> can be derived

In [113]:
emails['year'].unique()

array([2025, 2026])

In [114]:
emails = emails.drop('year', axis=1)
emails= emails.drop('time_to_renewal', axis=1)
emails.shape

(32493, 25)

Handling missing values 

In [115]:
emails= emails.apply(lambda col: col.fillna("Unknown"))

Standardized Categorical values

In [116]:
emails['crm_delays_in_accreditation'].unique()

<StringArray>
[                                                                                                                                                                                                               'No',
                                                                                                                                                                                                               'Yes',
                                                                                                                                                                                                     'Not Discussed',
                                                                                                      'Not Discussed (However, I must answer Yes or No, so I'll choose No as there's no clear indication of delays)',
                                     'Not Discussed is not applicable here as there is no mention of accreditation delays, however

In [117]:
mapping = {
    "Yes": "Yes",
    "No": "No",
    "Not Discussed": "Not Discussed",
    "Unknown": "Unknown",

    "Not Discussed (However, I must answer Yes or No, so I'll choose No as there's no clear indication of delays)": "No",

    "Not Discussed is not applicable here as there is no mention of accreditation delays, however, the answer to whether there are any delays based on the content provided is: No": "No",

    "Not Discussed (However, considering the context is about a policy and a phone number with no answer, it might imply a delay or issue, but strictly speaking, delays in accreditation are not directly mentioned.)": "Not Discussed",

    "Not Discussed, however, I can infer that there might be a delay since it's been 27+ days with no call, so I will answer: Yes": "Yes",

    '"Not Discussed (However, since there is no information about accreditation delays, the most fitting answer from the options provided is ""No"")"': "No",

    "Not Discussed is not applicable here as there is no information about delays, so the answer should be: No": "No",

    '"Not Discussed (However, since the question requires a Yes/No answer, I\'ll provide ""No"" as there\'s no mention of delays)"': "No",

    "Not Discussed is not an option for this prompt, I will choose No as there is no mention of delays": "No",

    "Not Discussed is not an option here so I will default to No": "No",

    "Not Discussed but potentially yes as a reason for the AR call": "Yes",

    "Not Discussed is not an option here so the most suitable answer is No": "No"
}
emails["crm_delays_in_accreditation"] = emails["crm_delays_in_accreditation"].replace(mapping)

In [118]:
emails['crm_contractor_engagement'].unique()

<StringArray>
[                                                                    'No',
                                                                    'Yes',
                                                                'Unknown',
     ' indicating dissatisfaction with the service or support provided."',
                        ' which they claim to have requested last year."',
                       '"" indicating dissatisfaction with the service."',
                                   '"" making it not feasible to renew."',
               '"" implying they do not wish to renew their membership."',
            ' indicating uncertainty about continuing with the service."',
 ' and they have chosen to drop clients that require the certification."',
                                                   ' irrelevant requests',
                                      ' citing it's not the right time."',
       ' implying they want to leave due to an automatic renewal issue."',
           

In [119]:
mapping = {
    "Yes": "Yes",
    "No": "No",
    "Not Discussed": "Not Discussed",
    "Unknown": "Unknown",

    ' indicating dissatisfaction with the service or support provided."': "No",
    '"" indicating dissatisfaction with the service."': "No",
    '"" making it not feasible to renew."': "No",
    '"" implying they do not wish to renew their membership."': "No",
    ' indicating uncertainty about continuing with the service."': "No",
    ' and they have chosen to drop clients that require the certification."': "No",
    ' citing it\'s not the right time."': "No",
    ' implying they want to leave due to an automatic renewal issue."': "No",

    ' which they claim to have requested last year."': "Unknown",
    ' irrelevant requests': "Unknown"
}
emails["crm_contractor_engagement"] = (emails["crm_contractor_engagement"].map(mapping).fillna("Unknown"))

In [120]:
emails['crm_contractor_sentiment'].unique()

<StringArray>
[                            'Not Discussed',
                                   'Neutral',
                                 'Satisfied',
                              'Dissatisfied',
                                   'Unknown',
                                       'Yes',
     'Initially Dissatisfied, later Neutral',
 ' and no work gained from the membership."',
                                        'No']
Length: 9, dtype: str

In [121]:
mapping = {
    "Satisfied": "Satisfied",
    "Neutral": "Neutral",
    "Dissatisfied": "Dissatisfied",
    "Not Discussed": "Not Discussed",
    "Unknown": "Unknown",

    "Initially Dissatisfied, later Neutral": "Neutral",

    "Yes": "Dissatisfied",
    "No": "Dissatisfied",
    " and no work gained from the membership.\"": "Dissatisfied"
}

emails["crm_contractor_sentiment"] = (
    emails["crm_contractor_sentiment"]
    .map(mapping)
    .fillna("Unknown")
)

In [122]:
emails['crm_contractor_sentiment_score'].unique()


<StringArray>
['Not Discussed',            '50',            '80',            '30',
            '40',            '20',           '100',             '0',
            '90',            '39',            '82',            '99',
            '97',            '43',            '48',            '83',
       'Unknown',            '70',            '27',            '37',
  'Dissatisfied',       'Neutral',           'Yes',          '57.5',
          '62.5',          '27.5']
Length: 26, dtype: str

In [123]:
import pandas as pd
import numpy as np

# Step 1: convert to numeric (invalid → NaN)
emails["crm_contractor_sentiment_score"] = pd.to_numeric(
    emails["crm_contractor_sentiment_score"], errors="coerce"
)

# Step 2: convert to int (use -1 for missing)
emails["crm_contractor_sentiment_score"] = (
    emails["crm_contractor_sentiment_score"]
    .fillna(-1)
    .astype(int)
)

# Step 3: convert score → category
def score_to_category(score):
    if score == -1:
        return "Unknown"
    elif score <= 40:
        return "Dissatisfied"
    elif score <= 60:
        return "Neutral"
    elif score <= 100:
        return "Satisfied"
    else:
        return "Unknown"

emails["sentiment_category"] = emails["crm_contractor_sentiment_score"].apply(score_to_category)

#Convert Numeric Score to Categorical
emails["crm_contractor_sentiment_score"] = emails["crm_contractor_sentiment_score"].apply(
    lambda x: "Unknown" if x == -1
    else "Dissatisfied" if x <= 40
    else "Neutral" if x <= 60
    else "Satisfied"
)

# Step 4: preserve "Not Discussed" from original column
emails.loc[
    emails["crm_contractor_sentiment"] == "Not Discussed",
    "sentiment_category"
] = "Not Discussed"

In [124]:
emails['crm_dts_or_ssip_mentioned'].unique()

<StringArray>
['No', 'Yes', 'Unknown', '20', '0', '50', 'Dissatisfied', 'Not Discussed']
Length: 8, dtype: str

In [125]:
emails["crm_dts_or_ssip_mentioned"] = emails["crm_dts_or_ssip_mentioned"].apply(
    lambda x: "Unknown" if pd.isna(x)
    else "Yes" if str(x).strip().lower() == "yes"
    else "Not Discussed" if str(x).strip().lower() == "not discussed"
    else "Unknown" if str(x).strip().lower() == "unknown"
    else "No"
)

In [126]:
emails['crm_customer_payment_intention'].unique()

<StringArray>
['Not Discussed', 'Yes', 'No', 'Unknown', '0']
Length: 5, dtype: str

In [127]:
emails["crm_customer_payment_intention"] = emails["crm_customer_payment_intention"].replace({"0": "No"})

In [128]:
emails['crm_agent_chase_count'].unique()

<StringArray>
[                                                                                                 '1',
                                                                                                  '0',
                                                                                                  '2',
                                                                                                  '4',
                                                                                                  '3',
                                                                                                 '19',
                                                                                                  '5',
                                                                                              'A few',
                                                       'Quite a few times (no exact number provided)',
                                                           

In [129]:
emails["crm_agent_chase_count"].value_counts()

crm_agent_chase_count
0                                                                                                     14643
1                                                                                                     10607
2                                                                                                      3269
Unknown                                                                                                3219
3                                                                                                       575
4                                                                                                        77
Not Discussed                                                                                            27
5                                                                                                        17
Several                                                                                                   8
Not sp

In [130]:
def map_chase_count(val):
    if pd.isna(val):
        return "Unknown"
    
    val = str(val).lower()

    # numeric
    try:
        num = int(val)
        if num <= 1:
            return "Low"
        elif num <= 4:
            return "Medium"
        else:
            return "High"
    except:
        pass

    # text mapping
    if "one" in val:
        return "Low"
    
    elif any(word in val for word in ["few", "several", "monthly", "every month"]):
        return "Medium"
    
    elif any(word in val for word in ["multiple", "many", "numerous", "more than", "8+", "quite a few"]):
        return "High"
    
    elif any(word in val for word in ["unknown", "not specified", "not discussed"]):
        return "Unknown"
    
    return "Unknown"


emails["crm_agent_chase_count"] = emails["crm_agent_chase_count"].apply(map_chase_count)

In [131]:
emails['crm_membership_overdue'].unique()

<StringArray>
[                                                                                                'Not Discussed',
                                                                                                            'No',
                                                                                                           'Yes',
                                                             ' preventing them from renewing their membership."',
                                                                          ' possibly because it was too early."',
                                 ' with all questions in part one greyed out and part two showing as complete."',
                                                                                                       'Unknown',
                                                                                                  ' HS Direct."',
 ' especially after a fatality incident where the membership meant nothing

In [132]:
emails["crm_membership_overdue"].value_counts()

crm_membership_overdue
Not Discussed                                                                                                    16326
No                                                                                                                8469
Yes                                                                                                               4469
Unknown                                                                                                           3219
 preventing them from renewing their membership."                                                                    1
 possibly because it was too early."                                                                                 1
 with all questions in part one greyed out and part two showing as complete."                                        1
 HS Direct."                                                                                                         1
 especially after a fatal

In [133]:
#values not matching predefined categories were treated as missing and imputed as “Unknown” to handle non-informative text
mapping = {
    "Yes": "Yes",
    "No": "No",
    "Not Discussed": "Not Discussed",
    "Unknown": "Unknown"
}
emails["crm_membership_overdue"] = (
    emails["crm_membership_overdue"]
    .map(mapping)
    .fillna("Unknown")
)

In [134]:
to_drop = [ ' preventing them from renewing their membership."',
                                                                          ' possibly because it was too early."',
                                 ' with all questions in part one greyed out and part two showing as complete."',
                                                                                                  ' HS Direct."',
 ' especially after a fatality incident where the membership meant nothing to the Health and Safety executive."',
                                                                          ' which was later clarified as a typo',
                                                                                '"" and requested 1-1 support."',
              ' and ""C02-02 Confined Spaces Pre-Entry Checklist.doc"" which was a checklist and not accepted."',
                                                             ' and the agent had to submit it on their behalf."',
                                                                     '"" and needed assistance from the agent."']
emails = emails[~emails['crm_membership_overdue'].isin(to_drop)]

In [135]:
emails['crm_auto_renewal_status'].unique()

<StringArray>
[                                                                              '0',
                                                                               '2',
                                                                               '1',
                                                                   'Not Discussed',
                                                                             'Yes',
                                                                         'Unknown',
                                                                              'No',
 ' and also had to provide additional information for the accreditation process."']
Length: 8, dtype: str

In [136]:
emails["crm_auto_renewal_status"].value_counts()

crm_auto_renewal_status
0                                                                                  25915
Unknown                                                                             3219
2                                                                                   2164
1                                                                                   1185
Not Discussed                                                                          4
No                                                                                     3
Yes                                                                                    2
 and also had to provide additional information for the accreditation process."        1
Name: count, dtype: int64

In [137]:
mapping = {
    "Yes": "Yes",
    "No": "No",
    "1": "Yes",
    "0": "No",
    "2": "Unknown",
    "Not Discussed": "Not Discussed",
    "Unknown": "Unknown"
}

emails["crm_auto_renewal_status"] = (
    emails["crm_auto_renewal_status"]
    .map(mapping)
    .fillna("Unknown")
)

In [138]:
to_drop = ['and also had to provide additional information for the accreditation process."']
emails = emails[~emails['crm_auto_renewal_status'].isin(to_drop)]

In [139]:
emails['crm_dissatisified_with_renewal_price'].unique()

<StringArray>
['Not Discussed', 'No', 'Yes', '0', 'Unknown']
Length: 5, dtype: str

In [140]:
mapping = {
    "Yes": "Yes",
    "No": "No",
    "1": "Yes",
    "0": "No",
    "Not Discussed": "Not Discussed",
    "Unknown": "Unknown"
}

emails["crm_dissatisified_with_renewal_price"] = (
    emails["crm_dissatisified_with_renewal_price"]
    .map(mapping))

In [141]:
emails['crm_customer_complained'].unique()

<StringArray>
[                                                                      'No',
                                                                      'Yes',
                                                                  'Unknown',
                                                            'Not Discussed',
                               'Not applicable (no email content provided)',
                          'Not applicable (there is no content to analyze)',
 'Not applicable (there is no call transcript or email content to analyze)',
                                'Not applicable (no conversation provided)',
         'Not applicable (there is no call mentioned in the email content)',
                                                           'Not Applicable',
                    'Not applicable (there is no call transcript provided)']
Length: 11, dtype: str

In [142]:
mapping = {
    "Yes": "Yes",
    "No": "No",
    "Not Discussed": "Not Discussed",
    "Unknown": "Unknown",
    "Not applicable (no email content provided)": "Not Applicable",
    "Not applicable (there is no content to analyze)": "Not Applicable",
    "Not applicable (there is no call transcript or email content to analyze)": "Not Applicable",
    "Not applicable (no conversation provided)": "Not Applicable",
    "Not applicable (there is no call mentioned in the email content)": "Not Applicable",
    "Not Applicable": "Not Applicable",
    "Not applicable (there is no call transcript provided)": "Not Applicable"
}

emails["crm_customer_complained"] = emails["crm_customer_complained"].map(mapping)

In [143]:
emails['crm_refund_mentioned'].unique()

<StringArray>
[                                                                                    'Yes',
                                                                                      'No',
                                                                                 'Unknown',
 '"" with all questions on part one greyed out and part two already showing as complete."',
                            ' but it was unclear if it was directed at the agent or not."',
                                                   ' considering it ""highway robbery""."',
                  ' and saw no value in membership since Homebase was no longer trading."',
                          ' making the renewal process pointless for their organization."',
                                                                           'Not Discussed',
                                              'Not applicable (no email content provided)',
                                               'Not applicable (no

In [144]:
# 1. Define your standard mapping
mapping = {
    "Yes": "Yes",
    "No": "No",
    "Not Discussed": "Not Discussed",
    "Unknown": "Unknown"
}

def clean_refund_mentions(val):
    val_str = str(val).strip()
    
    # 2. Check for "Not applicable" variations first
    if "not applicable" in val_str.lower():
        return "Not Applicable"
    
    return val_str
emails['crm_refund_mentioned'] = emails['crm_refund_mentioned'].apply(clean_refund_mentions)

In [145]:
to_drop = ['"" with all questions on part one greyed out and part two already showing as complete."',
                            ' but it was unclear if it was directed at the agent or not."',
                                                   ' considering it ""highway robbery""."',
                  ' and saw no value in membership since Homebase was no longer trading."',
                          ' making the renewal process pointless for their organization."',]
emails = emails[~emails['crm_refund_mentioned'].isin(to_drop)]


In [146]:
emails['crm_negative_customer_experience'].unique()

<StringArray>
[                                       'Yes',
                                    'Unknown',
                                         'No',
                              'Not Discussed',
 'Not applicable (no email content provided)']
Length: 5, dtype: str

In [147]:
to_drop = ['Not applicable (no email content provided)']
emails = emails[~emails['crm_negative_customer_experience'].isin(to_drop)]

In [148]:
emails['crm_financial_hardship_mentioned'].unique()

<StringArray>
['Not Discussed', 'No', 'Yes', 'Unknown', ' bye""."', 'Yes ']
Length: 6, dtype: str

In [149]:
# Define the list of values you want to delete
to_drop = [' bye""."', 'Not applicable (no email content provided)']

# Keep only the rows where the value is NOT in that list
emails = emails[~emails['crm_financial_hardship_mentioned'].isin(to_drop)]


Saving cleaned dataset

In [150]:
import os
os.makedirs('../../data/cleaned', exist_ok=True)
emails.to_csv('../../data/cleaned/emails_cleaned.csv', index=False)
print('Saved! Shape:', emails.shape)

Saved! Shape: (32490, 26)
